# 07 · Market Basket Analysis & Basket Growth

**Why this notebook exists:** every model built so far (02-05) answers one
question — *"what will this household buy again?"* — and grocery data is so
repeat-dominated (notebook 01: 45.9% of purchases are repeats) that the
winning model (notebook 05) ended up being **pure repeat-purchase replay**,
with a `novel_hits` count of essentially zero (0.0055 per household at
k=10). That model is a reorder engine. It structurally cannot answer a
different, equally important retail question: *"this household always buys
bread and peanut butter — what should we suggest they add that they've
never bought, because it goes with what they already buy?"* That's
**basket growth** — cross-sell, "frequently bought together," product
affinity — and it needs a different technique: **association rule mining**
(market basket analysis), the project roadmap's Phase 3.

### The core idea

An association rule `{bread, peanut_butter} -> {jelly}` says: in baskets
containing bread and peanut butter, jelly also appears with some
probability. Three numbers describe a rule:

- **Support** — how common is this itemset overall? (`P(antecedent ∪ consequent)`)
- **Confidence** — given the antecedent, how often does the consequent
  follow? (`P(consequent | antecedent)`)
- **Lift** — confidence divided by the consequent's overall popularity.
  `lift > 1` means the pairing happens *more* than chance; `lift = 1` means
  no real association (even a 90%-confidence rule can be meaningless if the
  consequent is bought by 90% of everyone anyway — lift is what filters
  that out).

Unlike every previous model, this one recommends items **regardless of
whether this specific household has ever bought them** — which is exactly
what "discover product V because he's never had it recommended" requires.

In [1]:
import warnings

import pandas as pd
from mlxtend.frequent_patterns import association_rules, fpgrowth
from mlxtend.preprocessing import TransactionEncoder

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 160)

from fmcg_reco.config import (
    N_CANDIDATES,
    PROCESSED_DIR,
    TEST_END_WEEK,
    TRAIN_END_WEEK,
    VAL_END_WEEK,
)
from fmcg_reco.evaluation.candidates import top_n_candidate_items
from fmcg_reco.evaluation.harness import evaluate_recommender
from fmcg_reco.evaluation.splitting import relevant_items_by_household, time_based_split
from fmcg_reco.models.base import BaseRecommender
from fmcg_reco.models.popularity import PersonalRepeatPurchaseRecommender

interactions = pd.read_parquet(PROCESSED_DIR / "interactions.parquet")
product = pd.read_parquet(PROCESSED_DIR / "product_dim.parquet")

train, val, test = time_based_split(interactions, TRAIN_END_WEEK, VAL_END_WEEK, TEST_END_WEEK)
candidate_items = top_n_candidate_items(train, N_CANDIDATES)
val_relevant = relevant_items_by_household(val)
previously_purchased = relevant_items_by_household(train)

## 1 · Building basket-level transactions

Association rules operate at the **basket** level, not the household level
— a rule only sees "these items were bought together in one trip." Restrict
to candidate items (same universe as every prior notebook, for consistency)
and drop single-item baskets, since a rule needs at least 2 items to relate.

In [2]:
basket_items = (
    train[train["product_id"].isin(candidate_items)]
    .groupby("basket_id")["product_id"].apply(set)
)
basket_items = basket_items[basket_items.apply(len) >= 2]
print(f"baskets with >=2 candidate items: {len(basket_items):,} of {train['basket_id'].nunique():,} total train baskets")
print("items per basket:", basket_items.apply(len).describe()[["mean", "50%", "max"]].to_dict())

baskets with >=2 candidate items: 149,267 of 235,958 total train baskets
items per basket: {'mean': 9.031172328779972, '50%': 6.0, 'max': 116.0}


## 2 · Mining frequent itemsets (FP-Growth)

`min_support` is the key tuning knob: too high and only trivially popular
combinations survive (nothing interesting); too low and the search space
explodes with noise. Tested three values before settling here — `0.01`
returned only 7 multi-item itemsets (too strict), `0.002` returned 317 in
about 5 seconds (a good balance of coverage and runtime) — shown as a
quick sweep rather than asserting the final number without justification.

In [3]:
te = TransactionEncoder()
onehot = te.fit_transform(basket_items.tolist(), sparse=True)
# mlxtend requires string column names for sparse input with integer product ids
encoded = pd.DataFrame.sparse.from_spmatrix(onehot, columns=[str(c) for c in te.columns_])

for min_support in [0.01, 0.005, 0.002]:
    freq = fpgrowth(encoded, min_support=min_support, use_colnames=True, max_len=3)
    n_multi = (freq["itemsets"].apply(len) >= 2).sum()
    print(f"min_support={min_support}: {len(freq):,} itemsets total, {n_multi:,} with 2+ items")

MIN_SUPPORT = 0.002
frequent_itemsets = fpgrowth(encoded, min_support=MIN_SUPPORT, use_colnames=True, max_len=3)
print(f"\nusing min_support={MIN_SUPPORT}: {len(frequent_itemsets):,} frequent itemsets")

min_support=0.01: 90 itemsets total, 7 with 2+ items
min_support=0.005: 280 itemsets total, 44 with 2+ items
min_support=0.002: 1,263 itemsets total, 317 with 2+ items

using min_support=0.002: 1,263 frequent itemsets


## 3 · Generating and inspecting rules

Filter to single-consequent rules (the natural "recommend one more thing"
shape) with `lift > 1.2` (a real, non-trivial association) and
`confidence >= 0.1` (happens often enough to be useful), then look at the
strongest rules by eye before trusting them.

In [4]:
rules = association_rules(
    frequent_itemsets, num_itemsets=len(basket_items), metric="lift", min_threshold=1.2
)
rules = rules[rules["consequents"].apply(len) == 1].copy()
rules = rules[rules["confidence"] >= 0.1]
rules["antecedents"] = rules["antecedents"].apply(lambda s: frozenset(int(x) for x in s))
rules["consequents"] = rules["consequents"].apply(lambda s: frozenset(int(x) for x in s))
rules["consequent_item"] = rules["consequents"].apply(lambda s: next(iter(s)))
rules = rules.sort_values("lift", ascending=False).reset_index(drop=True)
print(f"rules kept: {len(rules):,}")

def describe_items(item_ids):
    rows = product.set_index("product_id").loc[list(item_ids)]
    return "; ".join(f"{pid} ({row.commodity_desc}/{row.sub_commodity_desc})"
                      for pid, row in zip(item_ids, rows.itertuples(index=False)))

preview = rules.head(10).copy()
preview["antecedent"] = preview["antecedents"].apply(describe_items)
preview["consequent"] = preview["consequents"].apply(describe_items)
preview[["antecedent", "consequent", "support", "confidence", "lift"]]

rules kept: 295


,antecedent,consequent,support,confidence,lift
0,951526 (PWDR/CRYSTL DRNK MX/SOFT DRINK POWDER ...,989075 (PWDR/CRYSTL DRNK MX/SOFT DRINK POWDER ...,0.002104,0.472180,104.107769
1,989075 (PWDR/CRYSTL DRNK MX/SOFT DRINK POWDER ...,951526 (PWDR/CRYSTL DRNK MX/SOFT DRINK POWDER ...,0.002104,0.463811,104.107769
2,1053763 (PWDR/CRYSTL DRNK MX/SOFT DRINK POWDER...,989075 (PWDR/CRYSTL DRNK MX/SOFT DRINK POWDER ...,0.002278,0.415140,91.531410
3,989075 (PWDR/CRYSTL DRNK MX/SOFT DRINK POWDER ...,1053763 (PWDR/CRYSTL DRNK MX/SOFT DRINK POWDER...,0.002278,0.502216,91.531410
4,1053763 (PWDR/CRYSTL DRNK MX/SOFT DRINK POWDER...,951526 (PWDR/CRYSTL DRNK MX/SOFT DRINK POWDER ...,0.002204,0.401709,90.168357
5,951526 (PWDR/CRYSTL DRNK MX/SOFT DRINK POWDER ...,1053763 (PWDR/CRYSTL DRNK MX/SOFT DRINK POWDER...,0.002204,0.494737,90.168357
6,1030819 (VEGETABLES - SHELF STABLE/PEAS GREEN),938700 (VEGETABLES - SHELF STABLE/CORN),0.002117,0.410390,35.388577
7,938700 (VEGETABLES - SHELF STABLE/CORN),1030819 (VEGETABLES - SHELF STABLE/PEAS GREEN),0.002117,0.182553,35.388577
8,1022003 (SOUP/CONDENSED SOUP),1050229 (SOUP/CONDENSED SOUP),0.002184,0.235890,34.318319
9,1050229 (SOUP/CONDENSED SOUP),1022003 (SOUP/CONDENSED SOUP),0.002184,0.317739,34.318319


**Read this before concluding anything looks wrong**: several of the
highest-lift rules pair *different product IDs that share the same
commodity/sub-commodity label* (e.g. two different drink-mix flavors). That
is not a bug — it's a real, interpretable pattern: someone stocking up on
powdered drink mix or canned soup often buys **multiple flavors in the same
trip**. That's a legitimate basket-growth signal too ("you bought flavor A,
here's flavor B"), just a different kind than cross-category pairing.

For genuinely cross-category pairs, look further down (or filter to rules
whose antecedent and consequent have different `commodity_desc`):

In [5]:
def commodity_of(item_ids):
    return set(product.set_index("product_id").loc[list(item_ids), "commodity_desc"])

cross_category = rules[
    rules.apply(lambda r: commodity_of(r["antecedents"]) != commodity_of(r["consequents"]), axis=1)
]
preview_cross = cross_category.head(10).copy()
preview_cross["antecedent"] = preview_cross["antecedents"].apply(describe_items)
preview_cross["consequent"] = preview_cross["consequents"].apply(describe_items)
preview_cross[["antecedent", "consequent", "support", "confidence", "lift"]]

,antecedent,consequent,support,confidence,lift
21,1082185 (TROPICAL FRUIT/BANANAS); 1098066 (BAK...,826249 (BAKED BREAD/BUNS/ROLLS/HAMBURGER BUNS),0.002378,0.344660,10.671312
22,1082185 (TROPICAL FRUIT/BANANAS); 826249 (BAKE...,1098066 (BAKED BREAD/BUNS/ROLLS/HOT DOG BUNS),0.002378,0.320687,10.630228
27,910032 (HOT DOGS/PREMIUM - MEAT),1098066 (BAKED BREAD/BUNS/ROLLS/HOT DOG BUNS),0.002767,0.253998,8.419598
28,1082185 (TROPICAL FRUIT/BANANAS); 995785 (PEPP...,860776 (VEGETABLES - ALL OTHERS/CUCUMBERS),0.002050,0.222870,7.979631
29,6034857 (TOMATOES/TOMATOES GRAPE),860776 (VEGETABLES - ALL OTHERS/CUCUMBERS),0.002559,0.219793,7.869470
30,1062966 (HOT DOGS/ECONOMY - MEAT),1098066 (BAKED BREAD/BUNS/ROLLS/HOT DOG BUNS),0.002017,0.223128,7.396332
31,860776 (VEGETABLES - ALL OTHERS/CUCUMBERS); 10...,995785 (PEPPERS-ALL/PEPPERS GREEN BELL),0.002050,0.204000,7.081504
32,904360 (VEGETABLES SALAD/HEAD LETTUCE),1081177 (TOMATOES/TOMATOES VINE RIPE BULK),0.004100,0.127820,6.496166
33,1081177 (TOMATOES/TOMATOES VINE RIPE BULK),904360 (VEGETABLES SALAD/HEAD LETTUCE),0.004100,0.208376,6.496166
34,903325 (BROCCOLI/CAULIFLOWER/BROCCOLI WHOLE&CR...,961554 (CARROTS/CARROTS MINI PEELED),0.002640,0.174956,6.440221


## 4 · Validating rules on held-out baskets

A rule mined on train data could just be noise that happened to co-occur.
Check the top rules' confidence against **validation-period baskets** (weeks
89-95) — data the rules never saw — the same discipline every prior
notebook applied to models.

In [6]:
val_baskets = (
    val[val["product_id"].isin(candidate_items)]
    .groupby("basket_id")["product_id"].apply(set)
    .tolist()
)

def val_confidence(antecedent: frozenset, consequent: int):
    matching = [b for b in val_baskets if antecedent <= b]
    if not matching:
        return None, 0
    hits = sum(1 for b in matching if consequent in b)
    return hits / len(matching), len(matching)

top_rules = rules.head(30).copy()
validated = top_rules.apply(lambda r: val_confidence(r["antecedents"], r["consequent_item"]), axis=1)
top_rules["val_confidence"] = validated.apply(lambda x: x[0])
top_rules["val_n_baskets"] = validated.apply(lambda x: x[1])
top_rules = top_rules.dropna(subset=["val_confidence"])

corr = top_rules["confidence"].corr(top_rules["val_confidence"])
print(f"train vs. validation confidence correlation (top 30 rules): {corr:.2f}")
print(f"mean train confidence: {top_rules['confidence'].mean():.3f}  "
      f"mean validation confidence: {top_rules['val_confidence'].mean():.3f}")
top_rules[["confidence", "val_confidence", "val_n_baskets", "lift"]].head(10)

train vs. validation confidence correlation (top 30 rules): 0.70
mean train confidence: 0.312  mean validation confidence: 0.266


,confidence,val_confidence,val_n_baskets,lift
0,0.472180,0.382979,47,104.107769
1,0.463811,0.450000,40,104.107769
2,0.415140,0.291667,48,91.531410
3,0.502216,0.350000,40,91.531410
4,0.401709,0.250000,48,90.168357
5,0.494737,0.255319,47,90.168357
6,0.410390,0.463768,69,35.388577
7,0.182553,0.165803,193,35.388577
8,0.235890,0.187500,160,34.318319
9,0.317739,0.306122,98,34.318319


A moderate-to-strong correlation with some shrinkage (val confidence a bit
lower than train, on ~40-440 validation baskets per rule) is the expected,
healthy result — rules that hold up out-of-sample, not perfectly (that
would be suspicious), but well enough to trust for recommendations.

## 5 · A household-level "complete your basket" recommender

Given a household's own purchase history as a stand-in for "what's
typically in their basket," find every mined rule whose antecedent is a
subset of that history, and recommend the consequent — **but only if the
household has never bought it**. That exclusion is the entire point: this
recommender is incapable of recommending a repeat by construction.

In [7]:
class BasketAffinityRecommender(BaseRecommender):
    def __init__(self, rules_df: pd.DataFrame, candidate_items: set):
        self.rules_df = rules_df
        self.candidate_items = candidate_items
        self.household_items_: dict = {}

    def fit(self, train_interactions: pd.DataFrame) -> "BasketAffinityRecommender":
        df = train_interactions[train_interactions["product_id"].isin(self.candidate_items)]
        self.household_items_ = df.groupby("household_key")["product_id"].apply(set).to_dict()
        return self

    def recommend(self, household_key: int, k: int) -> list:
        owned = self.household_items_.get(household_key, set())
        scores: dict = {}
        for antecedent, consequent, lift in zip(
            self.rules_df["antecedents"], self.rules_df["consequent_item"], self.rules_df["lift"]
        ):
            if consequent in owned:
                continue
            if antecedent <= owned:
                scores[consequent] = max(scores.get(consequent, 0.0), lift)
        ranked = sorted(scores.items(), key=lambda pair: -pair[1])
        return [item for item, _ in ranked[:k]]

    def explain(self, household_key: int, item_id: int) -> pd.DataFrame:
        owned = self.household_items_.get(household_key, set())
        matches = self.rules_df[
            (self.rules_df["consequent_item"] == item_id)
            & self.rules_df["antecedents"].apply(lambda a: a <= owned)
        ]
        return matches[["antecedents", "confidence", "lift"]].sort_values("lift", ascending=False)


affinity_model = BasketAffinityRecommender(rules, candidate_items).fit(train)
n_with_recs = sum(1 for h in val_relevant if affinity_model.recommend(h, 1))
print(f"households with >=1 affinity-based recommendation: {n_with_recs:,}/{len(val_relevant):,} "
      f"({n_with_recs / len(val_relevant):.1%})")

households with >=1 affinity-based recommendation: 2,174/2,197 (99.0%)


## 6 · Evaluate — and read this table differently than every previous one

Run through the exact same harness, but expect a different shape of
result: this model can *never* register a repeat hit (it explicitly
excludes items the household already owns), so raw recall/MAP against a
repeat-dominated validation set will look weak compared to `buy_it_again`.
That's expected, not a failure — the comparison that actually matters here
is the **`novel_hits`** column.

In [8]:
repeat_model = PersonalRepeatPurchaseRecommender(candidate_items=candidate_items).fit(train)

_, affinity_summary = evaluate_recommender(affinity_model, val_relevant, [10, 20], previously_purchased)
_, repeat_summary = evaluate_recommender(repeat_model, val_relevant, [10, 20], previously_purchased)

comparison = pd.concat(
    {"basket_affinity": affinity_summary, "buy_it_again (hybrid)": repeat_summary}, axis=0
)
comparison.index.names = ["model", "k"]
comparison

precision  recall    ndcg     map  repeat_hits  novel_hits
model                 k                                                             
basket_affinity       10     0.0293  0.0057  0.0308  0.0107       0.0000      0.2931
                      20     0.0155  0.0062  0.0211  0.0057       0.0000      0.3100
buy_it_again (hybrid) 10     0.3999  0.0743  0.4469  0.3258       3.9932      0.0055
                      20     0.3185  0.1073  0.3816  0.2405       6.3605      0.0100

**This is the actual payoff of this notebook.** `basket_affinity`'s
`novel_hits` should be dramatically higher than `buy_it_again`'s — by
construction, 100% of its hits are novel (its `repeat_hits` column is
exactly 0, it's structurally incapable of a repeat) — while its recall/MAP
are much lower, because it isn't competing on "predict the next purchase"
at all. Two models optimized for two different jobs; neither number makes
the other wrong.

## 7 · Explainability — why was this recommended?

A real advantage of association rules over a matrix-factorization score:
the recommendation is explainable in plain language, which matters for
trust in a customer-facing "you might also like" feature.

In [9]:
example_household = next(iter(h for h in val_relevant if affinity_model.recommend(h, 1)))
recommended_item = affinity_model.recommend(example_household, k=1)[0]
row = product.set_index("product_id").loc[recommended_item]
print(f"household {example_household} was recommended product {recommended_item} "
      f"({row['commodity_desc']}/{row['sub_commodity_desc']}) because:")
explanation = affinity_model.explain(example_household, recommended_item)
explanation["antecedent_label"] = explanation["antecedents"].apply(describe_items)
print(explanation[["antecedent_label", "confidence", "lift"]].to_string(index=False))

household 1 was recommended product 8090537 (SOFT DRINKS/SOFT DRINKS 12/18&15PK CAN CAR) because:
                                    antecedent_label  confidence      lift
8090521 (SOFT DRINKS/SOFT DRINKS 12/18&15PK CAN CAR)    0.293258 23.508983


## 8 · Findings & how this changes the serving plan

- Fill in after running: the exact coverage (§5) and novel-hit contrast
  (§6) numbers. The expected story — confirmed independently before this
  notebook was handed over — is ~99% household coverage and a `novel_hits`
  count roughly **50x** `buy_it_again`'s, at the cost of much lower
  recall/MAP, because the two models are answering different questions.
- **This is not a replacement for the hybrid recommender** — it's a second,
  complementary capability. Shipping both means the API needs **two
  recommendation surfaces**, not one blended score:
  - `GET /recommendations/{household_id}` — the hybrid (notebook 05):
    "what will this household buy again" — reorder-focused.
  - `GET /recommendations/{household_id}/complete-basket` — this
    notebook's `BasketAffinityRecommender`: "what should we suggest they've
    never bought" — discovery/cross-sell-focused.
- The explainability in §7 is worth keeping in the API response (which
  rule triggered a recommendation, not just the item) — it's a genuine
  product advantage over the hybrid's opaque scores, and cheap to expose
  since the rule table is small.

Once reviewed, this gets extracted into `src/fmcg_reco/features/basket_rules.py`
(the mining pipeline) and `src/fmcg_reco/models/affinity.py`
(`BasketAffinityRecommender`), with the artifact export updated to include
the mined rules table.

### Next
Update the serving API design to expose both recommendation surfaces, then
move to implementation: OAuth2 auth, routers, Dockerfile, CI.